In [3]:
# -----######-----###### COPY ➜ VERIFY ➜ (OPTIONAL) DELETE FROM PLAYLIST -----######-----###### #
import os
import sys
import shutil
import hashlib
import pandas as pd
from tqdm import tqdm

# ===== helpers (no ASCII banner for sub-fns) =====
def _md5_of_file(path, chunk_size=1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

def _verify_pair(src, dst, mode="size"):
    try:
        if not (os.path.isfile(src) and os.path.isfile(dst)):
            return False
        if mode == "size":
            return os.path.getsize(src) == os.path.getsize(dst)
        elif mode == "md5":
            return _md5_of_file(src) == _md5_of_file(dst)
        else:
            return os.path.getsize(src) == os.path.getsize(dst)
    except Exception:
        return False

# -----######-----###### MAIN IMPORTABLE FUNCTION -----######-----###### #
def _copy_1308_playlistdel_GET_df_log(
    txt_path,
    dest_folder,
    verify_mode="size",      # "size" (fast, default) or "md5" (safer, slower)
    prompt_delete=True,      # asks once after copying; type YES to delete verified originals
    save_logs=True           # writes CSV logs next to the TXT
):
    """
    Copy all files in a UTF-16 tab-separated playlist (expects 'Location' column) to dest_folder,
    verify each copy, then optionally prompt to delete originals that verified OK.

    Returns
    -------
    pd.DataFrame with columns: Original_Path, Copied_Path, status, msg
    """
    # TQM
    tqdm.pandas()

    # Load playlist
    df = pd.read_csv(
        txt_path,
        sep="\t",
        encoding="utf-16",
        engine="python",
        on_bad_lines="skip"
    )
    df.columns = df.columns.str.strip()
    if 'Location' not in df.columns:
        raise ValueError("Playlist is missing required 'Location' column.")

    # Prepare
    os.makedirs(dest_folder, exist_ok=True)
    records = []

    # COPY (TQM)
    for i, row in tqdm(df.iterrows(), total=len(df), desc="Copying files"):
        original_path = str(row.get('Location', '')).strip()
        if not original_path or not os.path.isfile(original_path):
            records.append({"Original_Path": original_path, "Copied_Path": None, "status": "missing", "msg": "Source file not found"})
            continue

        dest_path = os.path.join(dest_folder, os.path.basename(original_path))
        try:
            shutil.copy2(original_path, dest_path)  # preserves metadata
            records.append({"Original_Path": original_path, "Copied_Path": dest_path, "status": "copied", "msg": ""})
        except Exception as e:
            records.append({"Original_Path": original_path, "Copied_Path": None, "status": "copy_error", "msg": str(e)})

    report = pd.DataFrame.from_records(records)

    # VERIFY (TQM)
    verify_mask = report['status'].eq('copied')
    if verify_mask.any():
        for idx in tqdm(report[verify_mask].index, desc="Verifying copies"):
            src = report.at[idx, 'Original_Path']
            dst = report.at[idx, 'Copied_Path']
            ok = _verify_pair(src, dst, mode=verify_mode)
            if ok:
                report.at[idx, 'status'] = 'verified'
            else:
                report.at[idx, 'status'] = 'verify_failed'
                report.at[idx, 'msg'] = f"Verification failed ({verify_mode})"

    # Logs before deletion
    base_dir = os.path.dirname(os.path.abspath(txt_path))
    if save_logs:
        report.to_csv(os.path.join(base_dir, "copy_report_pre_delete.csv"), index=False)
        report.loc[report['status'].eq('missing'), ['Original_Path']].rename(
            columns={'Original_Path': 'Missing_Files'}
        ).to_csv(os.path.join(base_dir, "missing_files_log.csv"), index=False)

    # PROMPT & DELETE (only verified)
    deleted_count = 0
    if prompt_delete:
        print("\n⚠️ COPY COMPLETE. Only 'verified' originals are eligible for deletion.")
        ans = input("Type EXACTLY 'YES' to delete verified originals from source: ").strip()
        if ans == "YES":
            del_mask = report['status'].eq('verified')
            for idx in tqdm(report[del_mask].index, desc="Deleting originals"):
                src = report.at[idx, 'Original_Path']
                try:
                    os.remove(src)
                    report.at[idx, 'status'] = 'deleted'
                    deleted_count += 1
                except Exception as e:
                    report.at[idx, 'status'] = 'delete_failed'
                    report.at[idx, 'msg'] = f"Delete error: {e}"
        else:
            print("🛑 Deletion canceled by user. Originals left untouched.")

    # Final logs
    if save_logs:
        report.to_csv(os.path.join(base_dir, "copy_report_final.csv"), index=False)
        if (report['status'] == 'copy_error').any():
            report.loc[report['status'].eq('copy_error')].to_csv(os.path.join(base_dir, "copy_errors_log.csv"), index=False)
        if (report['status'] == 'verify_failed').any():
            report.loc[report['status'].eq('verify_failed')].to_csv(os.path.join(base_dir, "verify_failed_log.csv"), index=False)

    print(f"\n✅ Done. Verified copies: {(report['status']=='verified').sum()} | Deleted: {deleted_count}")
    return report


In [4]:
# Paths you gave me:
txt_path = "/Users/yerik/Downloads/nnn.txt"
dest_folder = "/Volumes/_RECENT_YODJ/ddddd"

# Copy ➜ verify ➜ then ASK you to type YES before deleting verified originals
df_report = _copy_1308_playlistdel_GET_df_log(
    txt_path=txt_path,
    dest_folder=dest_folder,
    verify_mode="size",     # change to "md5" if you want cryptographic verification
    prompt_delete=True,
    save_logs=True
)

# optional: inspect summary
# print(df_report['status'].value_counts())
# df_report.to_csv("/Volumes/_RECENT_YODJ/copy_report_latest.csv", index=False)


Verifying copies: 100%|███████████████████████████████████████████████████████| 4/4 [00:00<00:00, 4688.99it/s]



⚠️ COPY COMPLETE. Only 'verified' originals are eligible for deletion.


KeyboardInterrupt: Interrupted by user